# 🌊 AI Urban Flood & Traffic Congestion Monitor (Official SYNEXAN)
### 100% Giao diện Gradio gốc ở Local (Tab 1, Tab 2, Tab 3 Dual Fusion)
Triển khai trên Kaggle GPU (Tesla T4) + Ngrok Static Domain: `hush-harmless-erased.ngrok-free.dev`.
Phục vụ cho Frontend trên Render.com.

In [ ]:
# 4. Khởi động Cloudflare Tunnel & Tự động kết nối trực tiếp về Render Frontend
import subprocess, time, os, re, urllib.request, json

# Địa chỉ chính thức của Web App trên Render
RENDER_FRONTEND_URL = "https://frontendflood.onrender.com"

!pkill -f cloudflared
!pkill -f "python app.py"

if not os.path.exists("/tmp/cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared

# Khởi động Cloudflare Tunnel trỏ vào Gradio port 7860
cf_cmd = ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860", "--logfile", "/tmp/cloudflared.log"]
cf_proc = subprocess.Popen(cf_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Chờ và đọc URL công khai từ file log Cloudflare
public_cf_url = None
for i in range(30):
    time.sleep(1)
    if os.path.exists("/tmp/cloudflared.log"):
        with open("/tmp/cloudflared.log", "r") as f:
            log_content = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
        if match:
            public_cf_url = match.group(0)
            break

print("=" * 70)
print("🚀 SYNEXAN AI FLOOD & TRAFFIC BACKEND ĐÃ SẴN SÀNG TRÊN GPU TESLA T4!")
if public_cf_url:
    print(f"🔥 CLOUDFLARE PUBLIC URL: {public_cf_url}")
    # Tự động gửi URL sang Render Frontend để tự động đồng bộ 100%
    for attempt in range(1, 4):
        try:
            req = urllib.request.Request(
                f"{RENDER_FRONTEND_URL}/api/backend-url",
                data=json.dumps({"url": public_cf_url}).encode("utf-8"),
                headers={"Content-Type": "application/json"}
            )
            resp = urllib.request.urlopen(req, timeout=10)
            print(f"✅ ĐÃ TỰ ĐỘNG ĐỒNG BỘ URL VỀ RENDER: {RENDER_FRONTEND_URL}")
            print("🎉 Giao diện người dùng sẽ tự động hiển thị mà KHÔNG CẦN F5 hay copy/paste.")
            break
        except Exception as e:
            print(f"⚠️ Thử gửi Render lần {attempt}/3: {e}")
            time.sleep(2)
else:
    print("⚠️ Đang thiết lập Cloudflare... Nếu chưa hiện URL, hãy thử chạy lại cell.")
print("=" * 70)

# Đảm bảo mã nguồn backend là mới nhất trước khi chạy
!git pull

# Chạy Backend Gradio App
subprocess.run(["python", "app.py"])


In [ ]:
# 2. Đồng bộ mã nguồn Backend mới nhất từ GitHub SYNEXAN/FrontendFlood
import os, sys
target_dir = "/kaggle/working/FrontendFlood"
if not os.path.exists(target_dir):
    !git clone https://github.com/SYNEXAN/FrontendFlood.git /kaggle/working/FrontendFlood
else:
    !cd /kaggle/working/FrontendFlood && git pull

os.chdir("/kaggle/working/FrontendFlood/backend")
sys.path.append("/kaggle/working/FrontendFlood/backend")
print("✅ Thư mục backend hiện tại:", os.getcwd())
!ls -lh

In [ ]:
# 3. Tự động tải Trọng số mô hình Deep Learning (Flood Seg & Vehicle Detection)
import os, urllib.request, time

os.makedirs("models", exist_ok=True)
models_info = [
    ("models/vietnam_flood_yolov8s_seg.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/vietnam_flood_yolov8s_seg.pt", 20000000),
    ("models/yolov8s_vehicle.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/yolov8s_vehicle.pt", 20000000)
]

for path, url, min_size in models_info:
    if os.path.exists(path) and os.path.getsize(path) >= min_size:
        print(f"✅ Mô hình {path} đã có sẵn ({os.path.getsize(path)} bytes).")
        continue
    for attempt in range(1, 4):
        try:
            print(f"⬇️ Đang tải {path} (lần {attempt}/3)...")
            if os.path.exists(path):
                os.remove(path)
            urllib.request.urlretrieve(url, path)
            if os.path.getsize(path) >= min_size:
                print(f"✅ Tải thành công {path}.")
                break
        except Exception as e:
            print(f"❌ Lỗi tải {path}: {e}")
            time.sleep(2)

print("✅ Sẵn sàng các file mô hình:")
!ls -lh models/

In [ ]:
# 4. Khởi động Cloudflare Tunnel & Tự động kết nối về Frontend Render\nimport os, sys, time, re, subprocess, requests\n\n# 🌐 Cấu hình địa chỉ Frontend trên Render của bạn\nRENDER_FRONTEND_URL = os.environ.get('RENDER_FRONTEND_URL', 'https://frontendflood.onrender.com')\n\n# (Tùy chọn) Nếu bạn có Cloudflare Zero Trust Tunnel Token (domain cố định):\n# CLOUDFLARE_TUNNEL_TOKEN = 'eyJh...token của bạn...'\nCLOUDFLARE_TUNNEL_TOKEN = os.environ.get('CLOUDFLARE_TUNNEL_TOKEN', '')\n\n!pkill -f cloudflared\ntime.sleep(1)\n\npublic_url = None\nif CLOUDFLARE_TUNNEL_TOKEN:\n    print('🚀 Đang khởi chạy Cloudflare Persistent Named Tunnel...')\n    proc = subprocess.Popen(['./cloudflared', 'tunnel', 'run', '--token', CLOUDFLARE_TUNNEL_TOKEN])\n    public_url = os.environ.get('CLOUDFLARE_PERSISTENT_DOMAIN', '')\nelse:\n    print('🚀 Đang khởi chạy Cloudflare Quick Tunnel (Tự động lấy URL mới)...')\n    proc = subprocess.Popen(\n        ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],\n        stdout=subprocess.PIPE,\n        stderr=subprocess.PIPE,\n        text=True\n    )\n    for _ in range(40):\n        line = proc.stderr.readline()\n        match = re.search(r'https://[a-zA-Z0-9-]+\\.trycloudflare\\.com', line)\n        if match:\n            public_url = match.group(0)\n            break\n        time.sleep(1)\n\nprint('=' * 70)\nprint(f'👉 CLOUDFLARE TUNNEL URL: {public_url}')\nprint('=' * 70)\n\n# 📡 TỰ ĐỘNG GỬI URL VỀ RENDER SERVER (Không cần làm thủ công)\nif public_url and RENDER_FRONTEND_URL:\n    try:\n        r = requests.post(\n            f'{RENDER_FRONTEND_URL.rstrip("/" )}/api/backend-url',\n            json={'url': public_url},\n            timeout=10\n        )\n        if r.status_code == 200:\n            print(f'✅ ĐÃ TỰ ĐỘNG ĐỒNG BỘ URL VỀ FRONTEND RENDER THÀNH CÔNG!')\n            print(f'👉 Giao diện người dùng sẽ tự động mở mà KHÔNG CẦN F5 hay copy/paste.')\n        else:\n            print(f'⚠️ Phản hồi từ Render ({r.status_code}): {r.text}')\n    except Exception as e:\n        print(f'ℹ️ Chưa kết nối được Render: {e}')\n        print(f'👉 Bạn có thể dán link {public_url} vào phần Cài đặt trên web.')\n\nsubprocess.run(['python', 'app.py'])